# Compare `build_dictionary` vs `extend_dictionary` · LeiData wav1 · windows 1–2

Matched knobs (PCA leading eigenvector, refine once at end):

| Knob | Value |
|---|---|
| windows | 2 (offset 0) |
| atoms / rounds | **20** |
| `eps_step_fraction` | 0.2 |
| `min_repeats` | 3 |
| `min_template_width` | `None` |
| `max_template_width` | 24 |
| `refine_iters` | 2 |
| `template_method` | `pca` |
| `disallow_overlap` | `True` |
| extend `min_coverage` | **0.9** |

Display: **`jet`** with 1st–99th percentile color limits (same as recent overlay notebook).

In [ ]:
%matplotlib inline

import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

REPO = Path("/home/yfreund/projects/gplrev3-webserver")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

LIBRARY_ROOT = Path("/home/yfreund/whale_processing")
DB_PATH = LIBRARY_ROOT / "result/dict_learning_catalog.sqlite"
WAV = LIBRARY_ROOT / "LeiData/SOCAL_H_76_230427_003345.x.wav"
BUILD_ROOT = LIBRARY_ROOT / "result/LeiData/dict_learning_templates/wav1_compare_build_pca_w2_r20"
EXTEND_ROOT = LIBRARY_ROOT / "result/LeiData/dict_learning_templates/wav1_compare_extend_pca_w2_r20"


def latest_run(root: Path) -> Path:
    stamped = sorted(
        [p for p in root.iterdir() if p.is_dir() and (p / "manifest.json").is_file()],
        key=lambda p: p.name,
    )
    if not stamped:
        raise FileNotFoundError(f"No stamped run under {root}")
    return stamped[-1]


def load_run(root: Path, label: str) -> dict:
    out_dir = latest_run(root)
    man = json.loads((out_dir / "manifest.json").read_text())
    params = man.get("params") or {}
    summary = {}
    sum_path = root / "summary.json"
    if sum_path.is_file():
        summary = json.loads(sum_path.read_text())
    e0, e1 = man.get("energy_before"), man.get("energy_after")
    frac = None
    if e0 is not None and e1 is not None and float(e0) > 0:
        frac = (float(e0) - float(e1)) / float(e0)
    templates = []
    for item in man.get("templates") or []:
        arr = np.load(out_dir / item["file"])
        view = arr[0] if arr.ndim == 3 else arr
        templates.append(
            {
                "template_id": int(item["template_id"]),
                "data": view,
                "meta": item,
                "source": item.get("source_segment_id") or getattr(
                    type("X", (), {}), "x", None
                ),
            }
        )
    # source may only live on npy sidecar / catalog; prefer manifest fields
    for t, item in zip(templates, man.get("templates") or []):
        t["source"] = item.get("source_segment_id")
        t["width"] = int(item.get("width") or t["data"].shape[-1])
    return {
        "label": label,
        "root": root,
        "out_dir": out_dir,
        "manifest": man,
        "params": params,
        "summary": summary,
        "dictionary_id": summary.get("dictionary_id"),
        "templates": templates,
        "matchings": list(man.get("matchings") or []),
        "n_atoms": len(templates),
        "n_matchings": len(man.get("matchings") or []),
        "stop_reason": man.get("stop_reason") or summary.get("stop_reason"),
        "energy_before": e0,
        "energy_after": e1,
        "fraction_removed": frac,
        "coverage": summary.get("coverage"),
    }


def clim_p1_p99(arr: np.ndarray) -> tuple[float, float]:
    vmin, vmax = np.percentile(arr, [1, 99])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
        return float(np.min(arr)), float(np.max(arr))
    return float(vmin), float(vmax)


build = load_run(BUILD_ROOT, "build")
extend = load_run(EXTEND_ROOT, "extend")
print("build :", build["out_dir"], "dict", build["dictionary_id"])
print("extend:", extend["out_dir"], "dict", extend["dictionary_id"])

## Summary table

In [ ]:
def pct(frac):
    return "—" if frac is None else f"{100 * frac:.1f}%"


lines = [
    "| | build | extend |",
    "|---|---:|---:|",
    f"| `dictionary_id` | {build['dictionary_id']} | {extend['dictionary_id']} |",
    f"| n_atoms | {build['n_atoms']} | {extend['n_atoms']} |",
    f"| n_matchings | {build['n_matchings']} | {extend['n_matchings']} |",
    f"| %E removed | {pct(build['fraction_removed'])} | {pct(extend['fraction_removed'])} |",
    f"| stop | `{build['stop_reason']}` | `{extend['stop_reason']}` |",
    f"| coverage | {build.get('coverage')} | {extend.get('coverage')} |",
]
display(Markdown("\n".join(lines)))

## Template pairwise correlation

In [ ]:
def corr(a: np.ndarray, b: np.ndarray) -> float | None:
    if a.shape != b.shape:
        return None
    av, bv = a.ravel() - a.mean(), b.ravel() - b.mean()
    denom = float(np.linalg.norm(av) * np.linalg.norm(bv))
    if denom <= 0:
        return None
    return float(np.dot(av, bv) / denom)


n = max(build["n_atoms"], extend["n_atoms"])
rows = [
    "| i | build id | extend id | same shape | corr | rel ‖Δ‖ |",
    "|---:|---:|---:|:---:|---:|---:|",
]
corrs = []
for i in range(n):
    bt = build["templates"][i] if i < build["n_atoms"] else None
    et = extend["templates"][i] if i < extend["n_atoms"] else None
    if bt is None or et is None:
        rows.append(f"| {i} | {None if bt is None else bt['template_id']} | "
                    f"{None if et is None else et['template_id']} | — | — | — |")
        continue
    c = corr(bt["data"], et["data"])
    if c is not None:
        corrs.append(c)
        rel = float(np.linalg.norm(bt["data"] - et["data"]) / (np.linalg.norm(bt["data"]) + 1e-12))
    else:
        rel = None
    rows.append(
        f"| {i} | {bt['template_id']} | {et['template_id']} | "
        f"{bt['data'].shape == et['data'].shape} | "
        f"{'—' if c is None else f'{c:.6f}'} | "
        f"{'—' if rel is None else f'{rel:.3e}'} |"
    )
identical = bool(corrs) and len(corrs) == n and all(abs(c - 1.0) < 1e-9 for c in corrs)
rows.append(f"| **identical** |  |  |  | **{identical}** |  |")
display(Markdown("\n".join(rows)))

## Templates side-by-side (`jet`, p1–p99)

In [ ]:
n = max(build["n_atoms"], extend["n_atoms"])
fig, axes = plt.subplots(n, 2, figsize=(8, 2.6 * n), squeeze=False)
for i in range(n):
    for j, run in enumerate((build, extend)):
        ax = axes[i][j]
        if i >= run["n_atoms"]:
            ax.axis("off")
            ax.set_title(f"{run['label']} · missing")
            continue
        t = run["templates"][i]
        view = t["data"]
        vmin, vmax = clim_p1_p99(view)
        im = ax.imshow(view, aspect="auto", origin="lower", cmap="jet", vmin=vmin, vmax=vmax)
        ax.set_title(f"{run['label']} tid={t['template_id']}  {view.shape[0]}×{t['width']}", fontsize=10)
        ax.set_ylabel("freq")
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
axes[-1][0].set_xlabel("time")
axes[-1][1].set_xlabel("time")
fig.suptitle("Templates · jet · clim p1–p99", y=1.01)
fig.tight_layout()
plt.show()

## Matching overlays on windows 1–2 (`jet`, p1–p99)

In [ ]:
from database import catalog


def load_windows(n: int = 2):
    with catalog.connect(DB_PATH) as conn:
        wav_row = catalog.find_wav_file(conn, str(WAV))
        segs = catalog.list_gpl_segmentations_for_wav(conn, wav_row["wav_id"])
        wins = catalog.list_windows_for_segmentation(
            conn, segs[0]["segmentation_id"], limit=n, offset=0
        )
        return [(w["window_id"], np.load(w["npy_path"])) for w in wins]


def widths_for(run: dict) -> dict[int, int]:
    return {t["template_id"]: t["width"] for t in run["templates"]}


windows = load_windows(2)
cmap = plt.cm.tab10

for run in (build, extend):
    by_win = defaultdict(list)
    for m in run["matchings"]:
        by_win[m["window_id"]].append(m)
    widths = widths_for(run)
    fig, axes = plt.subplots(len(windows), 1, figsize=(14, 3.2 * len(windows)), squeeze=False)
    for ax, (wid, spec) in zip(axes[:, 0], windows):
        vmin, vmax = clim_p1_p99(spec)
        ax.imshow(spec, aspect="auto", origin="lower", cmap="jet", vmin=vmin, vmax=vmax)
        drawn = 0
        for m in by_win.get(wid, []):
            tid = int(m["template_id"])
            t0 = int(m["t_start"])
            w = widths.get(tid, 1)
            ax.axvspan(t0, t0 + w, color="white", alpha=0.22)
            ax.axvline(t0, color=cmap(tid % 10), lw=1.2, alpha=0.95)
            ax.axvline(t0 + w, color=cmap(tid % 10), lw=1.0, alpha=0.7, ls="--")
            drawn += 1
        ax.set_title(
            f"{run['label']} · {wid[-40:]} · {drawn} matchings · clim p1–p99 [{vmin:.3g}, {vmax:.3g}]"
        )
        ax.set_ylabel("freq")
    axes[-1, 0].set_xlabel("time bin")
    fig.suptitle(f"{run['label']} overlays · jet", y=1.01)
    fig.tight_layout()
    plt.show()

## Matchings per atom

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for ax, run in zip(axes, (build, extend)):
    counts = Counter(int(m["template_id"]) for m in run["matchings"])
    tids = sorted(counts) if counts else []
    ax.bar([str(t) for t in tids], [counts[t] for t in tids], color="steelblue")
    ax.set_title(f"{run['label']} · {run['n_matchings']} matchings")
    ax.set_xlabel("template_id")
    ax.set_ylabel("# matchings")
    for tid in tids:
        ax.text(str(tid), counts[tid], str(counts[tid]), ha="center", va="bottom")
fig.tight_layout()
plt.show()